# Module 3: Random Effects vs Fixed Effects

## Learning Objectives

By completing this notebook, you will:
1. Compare RE and FE estimators empirically
2. Implement the Hausman specification test
3. Understand the bias-efficiency trade-off
4. Make informed choices between RE and FE
5. Recognize when each estimator is appropriate

## Prerequisites

- Module 2: Fixed Effects (completed)
- Module 3.1: RE Implementation (completed)
- Understanding of hypothesis testing

## Estimated Time

75-90 minutes

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, compare

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed
np.random.seed(42)

print("✅ Setup complete")

## 1. The Key Trade-Off: Bias vs Efficiency

### Fixed Effects (FE)
- **Advantage:** Consistent even if $Cov(\alpha_i, X_{it}) \neq 0$
- **Disadvantage:** Less efficient (uses only within variation)
- **Cannot estimate:** Time-invariant variables

### Random Effects (RE)
- **Advantage:** More efficient (uses within + between variation)
- **Advantage:** Can estimate time-invariant variables
- **Disadvantage:** Biased if $Cov(\mu_i, X_{it}) \neq 0$

### The Question

**When is RE valid?**

Test: $H_0: Cov(\mu_i, X_{it}) = 0$ (RE is consistent)

This is the **Hausman test**.

## 2. Generate Two Datasets

### Dataset 1: RE Valid
Entity effects **uncorrelated** with X

### Dataset 2: RE Invalid
Entity effects **correlated** with X (endogeneity)

In [ ]:
def generate_panel_data(n_entities=100, n_periods=10, 
                        correlation=0.0, beta_true=1.5):
    """
    Generate panel data with controlled correlation.
    
    Parameters
    ----------
    n_entities : int
    n_periods : int
    correlation : float
        Correlation between entity effects and X
        0 = RE valid, >0 = RE invalid
    beta_true : float
        True causal effect
    
    Returns
    -------
    pd.DataFrame
    """
    # Entity effects
    mu_i = np.random.randn(n_entities) * 10
    
    # Generate X with controlled correlation to mu_i
    X = []
    for i in range(n_entities):
        # Base X depends on mu_i (creates correlation)
        base_x = 20 + correlation * mu_i[i] + np.random.randn() * 5
        
        X_i = []
        for t in range(n_periods):
            # X varies around base
            X_i.append(base_x + np.random.randn() * 3)
        X.extend(X_i)
    
    X = np.array(X)
    
    # Generate y
    mu_repeated = np.repeat(mu_i, n_periods)
    epsilon = np.random.randn(n_entities * n_periods) * 5
    y = 50 + beta_true * X + mu_repeated + epsilon
    
    # Create DataFrame
    entities = np.repeat(np.arange(1, n_entities + 1), n_periods)
    periods = np.tile(np.arange(1, n_periods + 1), n_entities)
    
    df = pd.DataFrame({
        'entity_id': entities,
        'time': periods,
        'y': y,
        'x': X
    })
    
    return df, mu_i

# Generate data with RE valid (no correlation)
df_valid, mu_valid = generate_panel_data(correlation=0.0)

# Generate data with RE invalid (strong correlation)
df_invalid, mu_invalid = generate_panel_data(correlation=0.5)

print("Generated Two Datasets:")
print("\n1. RE VALID: Cov(mu_i, X) = 0")
print("   Entity effects uncorrelated with X")
print("\n2. RE INVALID: Cov(mu_i, X) ≠ 0")
print("   Entity effects correlated with X (endogeneity)")

### Visualize the Correlation

In [ ]:
# Compute entity-level means of X
entity_means_valid = df_valid.groupby('entity_id')['x'].mean()
entity_means_invalid = df_invalid.groupby('entity_id')['x'].mean()

# Correlations
corr_valid = np.corrcoef(mu_valid, entity_means_valid)[0, 1]
corr_invalid = np.corrcoef(mu_invalid, entity_means_invalid)[0, 1]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: RE valid
axes[0].scatter(entity_means_valid, mu_valid, alpha=0.5, s=50)
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Mean of X (by entity)', fontsize=11)
axes[0].set_ylabel('Entity Effect (mu_i)', fontsize=11)
axes[0].set_title(f'RE VALID: Correlation = {corr_valid:.3f}', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Panel 2: RE invalid
axes[1].scatter(entity_means_invalid, mu_invalid, alpha=0.5, s=50, color='red')
axes[1].axhline(y=0, color='blue', linestyle='--', linewidth=1)

# Add regression line
z = np.polyfit(entity_means_invalid, mu_invalid, 1)
p = np.poly1d(z)
axes[1].plot(entity_means_invalid, p(entity_means_invalid), 
             'k-', linewidth=2, alpha=0.5, label='Best fit')

axes[1].set_xlabel('Mean of X (by entity)', fontsize=11)
axes[1].set_ylabel('Entity Effect (mu_i)', fontsize=11)
axes[1].set_title(f'RE INVALID: Correlation = {corr_invalid:.3f}', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Left: mu_i uncorrelated with X → RE is consistent")
print("Right: mu_i correlated with X → RE is biased, FE needed")

## 3. Compare Estimators: RE Valid Case

In [ ]:
# Prepare data
df_valid_panel = df_valid.set_index(['entity_id', 'time'])

# Fixed Effects
model_fe_valid = PanelOLS(
    dependent=df_valid_panel['y'],
    exog=df_valid_panel[['x']],
    entity_effects=True
)
results_fe_valid = model_fe_valid.fit(cov_type='clustered', cluster_entity=True)

# Random Effects
model_re_valid = RandomEffects(
    dependent=df_valid_panel['y'],
    exog=df_valid_panel[['x']]
)
results_re_valid = model_re_valid.fit()

# Compare
print("RE VALID CASE: Estimates When RE Assumptions Hold")
print("=" * 70)
print(f"{'Estimator':<20} {'Beta':>12} {'Std Err':>12} {'t-stat':>10}")
print("=" * 70)

beta_fe_valid = results_fe_valid.params['x']
se_fe_valid = results_fe_valid.std_errors['x']
t_fe_valid = results_fe_valid.tstats['x']

beta_re_valid = results_re_valid.params['x']
se_re_valid = results_re_valid.std_errors['x']
t_re_valid = results_re_valid.tstats['x']

print(f"{'True value':<20} {1.5:>12.4f} {'':>12} {'':>10}")
print(f"{'Fixed Effects':<20} {beta_fe_valid:>12.4f} {se_fe_valid:>12.4f} {t_fe_valid:>10.2f}")
print(f"{'Random Effects':<20} {beta_re_valid:>12.4f} {se_re_valid:>12.4f} {t_re_valid:>10.2f}")

print("\n" + "=" * 70)
print(f"Bias from true (1.5):")
print(f"  FE: {abs(beta_fe_valid - 1.5):.4f}")
print(f"  RE: {abs(beta_re_valid - 1.5):.4f}")

efficiency_gain = (se_fe_valid - se_re_valid) / se_fe_valid * 100
print(f"\nEfficiency gain (RE vs FE): {efficiency_gain:.1f}%")

print("\n✅ Both estimators are consistent (unbiased)")
print("✅ RE is more efficient (smaller SE)")

## 4. Compare Estimators: RE Invalid Case

In [ ]:
# Prepare data
df_invalid_panel = df_invalid.set_index(['entity_id', 'time'])

# Fixed Effects
model_fe_invalid = PanelOLS(
    dependent=df_invalid_panel['y'],
    exog=df_invalid_panel[['x']],
    entity_effects=True
)
results_fe_invalid = model_fe_invalid.fit(cov_type='clustered', cluster_entity=True)

# Random Effects
model_re_invalid = RandomEffects(
    dependent=df_invalid_panel['y'],
    exog=df_invalid_panel[['x']]
)
results_re_invalid = model_re_invalid.fit()

# Compare
print("RE INVALID CASE: Estimates When RE Assumptions Violated")
print("=" * 70)
print(f"{'Estimator':<20} {'Beta':>12} {'Std Err':>12} {'t-stat':>10}")
print("=" * 70)

beta_fe_invalid = results_fe_invalid.params['x']
se_fe_invalid = results_fe_invalid.std_errors['x']
t_fe_invalid = results_fe_invalid.tstats['x']

beta_re_invalid = results_re_invalid.params['x']
se_re_invalid = results_re_invalid.std_errors['x']
t_re_invalid = results_re_invalid.tstats['x']

print(f"{'True value':<20} {1.5:>12.4f} {'':>12} {'':>10}")
print(f"{'Fixed Effects':<20} {beta_fe_invalid:>12.4f} {se_fe_invalid:>12.4f} {t_fe_invalid:>10.2f}")
print(f"{'Random Effects':<20} {beta_re_invalid:>12.4f} {se_re_invalid:>12.4f} {t_re_invalid:>10.2f}")

print("\n" + "=" * 70)
print(f"Bias from true (1.5):")
print(f"  FE: {abs(beta_fe_invalid - 1.5):.4f}")
print(f"  RE: {abs(beta_re_invalid - 1.5):.4f}")

print("\n✅ FE remains consistent (unbiased)")
print("⚠️  RE is BIASED due to endogeneity!")
print("\n→ When in doubt, use FE")

### Visualize the Bias

In [ ]:
# Create comparison plot
fig, ax = plt.subplots(figsize=(10, 6))

estimates = ['True Value', 'FE (Valid)', 'RE (Valid)', 'FE (Invalid)', 'RE (Invalid)']
values = [
    1.5,
    beta_fe_valid,
    beta_re_valid,
    beta_fe_invalid,
    beta_re_invalid
]
colors = ['green', 'blue', 'blue', 'blue', 'red']

y_pos = np.arange(len(estimates))

bars = ax.barh(y_pos, values, color=colors, alpha=0.6, edgecolor='black')

# True value line
ax.axvline(x=1.5, color='green', linestyle='--', linewidth=2, 
           label='True value', alpha=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels(estimates)
ax.set_xlabel('Estimated Beta', fontsize=12)
ax.set_title('Comparison: FE vs RE Under Different Conditions', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, v in enumerate(values):
    ax.text(v, i, f' {v:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("Key observation: RE is biased when Cov(mu_i, X) ≠ 0")
print("                FE is always consistent")

## 5. The Hausman Test

### Intuition

**If RE assumptions hold:**
- Both FE and RE are consistent
- Estimates should be similar
- Differences due to sampling variation only

**If RE assumptions violated:**
- FE is consistent
- RE is biased
- Systematic difference between estimates

### Test Statistic

$$H = (\hat{\beta}_{FE} - \hat{\beta}_{RE})' [Var(\hat{\beta}_{FE}) - Var(\hat{\beta}_{RE})]^{-1} (\hat{\beta}_{FE} - \hat{\beta}_{RE})$$

Under $H_0$: $H \sim \chi^2_k$

### Decision Rule

- **p-value > 0.05:** Cannot reject $H_0$ → Use RE (more efficient)
- **p-value < 0.05:** Reject $H_0$ → Use FE (RE is biased)

In [ ]:
def hausman_test_manual(beta_fe, beta_re, var_fe, var_re):
    """
    Implement Hausman test manually.
    
    Parameters
    ----------
    beta_fe : float
        FE estimate
    beta_re : float
        RE estimate
    var_fe : float
        Variance of FE estimate
    var_re : float
        Variance of RE estimate
    
    Returns
    -------
    dict
        Test statistic and p-value
    """
    # Difference in estimates
    diff = beta_fe - beta_re
    
    # Variance of difference
    var_diff = var_fe - var_re
    
    # Hausman statistic
    if var_diff > 0:  # Should be positive
        H = (diff**2) / var_diff
    else:
        H = np.nan
        
    # P-value (chi-squared with df=1 for single coefficient)
    p_value = 1 - stats.chi2.cdf(H, df=1)
    
    return {
        'statistic': H,
        'p_value': p_value,
        'df': 1
    }

# Test on valid data
print("Hausman Test Results")
print("=" * 70)
print("\n1. RE VALID Case:")
print("-" * 70)

hausman_valid = hausman_test_manual(
    beta_fe_valid, beta_re_valid,
    se_fe_valid**2, se_re_valid**2
)

print(f"H0: RE assumptions hold (Cov(mu_i, X) = 0)")
print(f"")
print(f"Difference (FE - RE):  {beta_fe_valid - beta_re_valid:.4f}")
print(f"Hausman statistic:     {hausman_valid['statistic']:.4f}")
print(f"Degrees of freedom:    {hausman_valid['df']}")
print(f"P-value:               {hausman_valid['p_value']:.4f}")

if hausman_valid['p_value'] > 0.05:
    print(f"\n✅ Cannot reject H0 (p = {hausman_valid['p_value']:.3f} > 0.05)")
    print("   → Use RANDOM EFFECTS (more efficient)")
else:
    print(f"\n⚠️  Reject H0 (p = {hausman_valid['p_value']:.3f} < 0.05)")
    print("   → Use FIXED EFFECTS (RE is biased)")

# Test on invalid data
print("\n\n2. RE INVALID Case:")
print("-" * 70)

hausman_invalid = hausman_test_manual(
    beta_fe_invalid, beta_re_invalid,
    se_fe_invalid**2, se_re_invalid**2
)

print(f"H0: RE assumptions hold (Cov(mu_i, X) = 0)")
print(f"")
print(f"Difference (FE - RE):  {beta_fe_invalid - beta_re_invalid:.4f}")
print(f"Hausman statistic:     {hausman_invalid['statistic']:.4f}")
print(f"Degrees of freedom:    {hausman_invalid['df']}")
print(f"P-value:               {hausman_invalid['p_value']:.4f}")

if hausman_invalid['p_value'] > 0.05:
    print(f"\n✅ Cannot reject H0 (p = {hausman_invalid['p_value']:.3f} > 0.05)")
    print("   → Use RANDOM EFFECTS (more efficient)")
else:
    print(f"\n⚠️  Reject H0 (p = {hausman_invalid['p_value']:.3f} < 0.05)")
    print("   → Use FIXED EFFECTS (RE is biased)")

## 6. Practical Decision Framework

### Workflow for Model Choice

```
1. Estimate both FE and RE
     ↓
2. Run Hausman test
     ↓
3. Decision:
     ↓
   p > 0.05 → Use RE (more efficient, assumptions hold)
     ↓
   p < 0.05 → Use FE (RE is biased)
```

### Additional Considerations

**Use FE if:**
- Hausman test rejects
- Theoretical reasons to expect endogeneity
- Within-entity effects are the research question
- Large T (efficiency gain from RE is small)

**Use RE if:**
- Hausman test fails to reject
- Need to estimate time-invariant variables
- Entities are random sample from population
- Small T (efficiency gain matters)

---

## Exercises

### Exercise 1: Simulate Hausman Test Power

**Task:** Investigate how correlation strength affects Hausman test power.

**Procedure:**
1. Generate data with varying correlation (0, 0.1, 0.2, 0.3, 0.4, 0.5)
2. For each, run 100 simulations
3. Compute rejection rate (power)
4. Plot power curve

**Expected:** Higher correlation → higher rejection rate

In [ ]:
# YOUR CODE HERE
def hausman_power_analysis(correlations, n_sims=100):
    """
    Analyze Hausman test power.
    
    Parameters
    ----------
    correlations : list
        Correlation levels to test
    n_sims : int
        Simulations per correlation
    
    Returns
    -------
    dict
        Rejection rates by correlation
    """
    # TODO: Implement power analysis
    pass

In [ ]:
# SOLUTION (hidden)
def hausman_power_analysis_solution(correlations, n_sims=100):
    """Analyze Hausman test power."""
    results = {}
    
    for corr in correlations:
        rejections = 0
        
        for sim in range(n_sims):
            # Generate data
            df_sim, _ = generate_panel_data(
                n_entities=50, n_periods=8, correlation=corr
            )
            df_sim_panel = df_sim.set_index(['entity_id', 'time'])
            
            try:
                # FE
                model_fe = PanelOLS(df_sim_panel['y'], df_sim_panel[['x']], 
                                    entity_effects=True)
                results_fe = model_fe.fit()
                
                # RE
                model_re = RandomEffects(df_sim_panel['y'], df_sim_panel[['x']])
                results_re = model_re.fit()
                
                # Hausman test
                beta_fe = results_fe.params['x']
                beta_re = results_re.params['x']
                var_fe = results_fe.std_errors['x']**2
                var_re = results_re.std_errors['x']**2
                
                hausman = hausman_test_manual(beta_fe, beta_re, var_fe, var_re)
                
                if hausman['p_value'] < 0.05:
                    rejections += 1
            except:
                pass
        
        results[corr] = rejections / n_sims
    
    return results

In [ ]:
# Test your function (WARNING: Takes ~1 minute)
def test_exercise_1():
    """Test Hausman power analysis."""
    print("Running power analysis (this may take a minute)...")
    
    correlations = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
    power_results = hausman_power_analysis(
        correlations, n_sims=50  # Reduced for speed
    )
    
    assert power_results is not None, "Function returns None - did you implement it?"
    assert len(power_results) == len(correlations), "Wrong number of results"
    
    print("\n✅ Correct! Power analysis complete.")
    print("\nHausman Test Power by Correlation:")
    print("=" * 50)
    for corr, power in power_results.items():
        print(f"  Correlation {corr:.1f}: Power = {power:.3f}")
    
    # Plot power curve
    fig, ax = plt.subplots(figsize=(10, 6))
    
    corrs = list(power_results.keys())
    powers = list(power_results.values())
    
    ax.plot(corrs, powers, marker='o', linewidth=2, markersize=10)
    ax.axhline(y=0.05, color='red', linestyle='--', linewidth=1.5,
               label='Size (alpha = 0.05)')
    ax.axhline(y=0.80, color='green', linestyle='--', linewidth=1.5,
               label='80% Power', alpha=0.5)
    
    ax.set_xlabel('Correlation (mu_i with X)', fontsize=12)
    ax.set_ylabel('Rejection Rate (Power)', fontsize=12)
    ax.set_title('Hausman Test Power Curve', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.show()
    
    print("\nInterpretation:")
    print("  - At correlation=0: Rejection rate ≈ 0.05 (correct size)")
    print("  - As correlation increases: Power increases")
    print("  - Test correctly detects RE violations")
    
    return True

# Uncomment to test (takes ~1 minute)
# test_exercise_1()

### Exercise 2: Compare Estimators on Real Economic Data

**Task:** Apply FE, RE, and Hausman test to a simulated economic dataset.

**Scenario:** Firm R&D and productivity
- Some firms have unobserved management quality
- Quality affects both R&D spending and productivity

**Questions:**
1. Estimate with FE and RE
2. Run Hausman test
3. Which estimator is appropriate?

In [ ]:
# Generate firm-level panel data
np.random.seed(123)
n_firms = 80
n_years = 12

# Management quality (unobserved)
quality = np.random.randn(n_firms) * 15

# R&D spending (correlated with quality)
rd_data = []
for i in range(n_firms):
    base_rd = 50 + 0.4 * quality[i]  # Correlation!
    for t in range(n_years):
        rd_data.append(base_rd + np.random.randn() * 8)

rd_data = np.array(rd_data)

# Productivity
quality_repeated = np.repeat(quality, n_years)
epsilon = np.random.randn(n_firms * n_years) * 10
productivity = 100 + 1.2 * rd_data + quality_repeated + epsilon

# Create DataFrame
firms = np.repeat(np.arange(1, n_firms + 1), n_years)
years = np.tile(np.arange(2010, 2010 + n_years), n_firms)

df_firms = pd.DataFrame({
    'firm_id': firms,
    'year': years,
    'productivity': productivity,
    'rd_spending': rd_data
})

print("Firm-Level Panel Data Generated")
print(f"  Firms: {n_firms}")
print(f"  Years: {n_years}")
print(f"  True causal effect of R&D: 1.2")
print(f"\nManagement quality is CORRELATED with R&D (endogeneity)")
print(f"→ RE assumptions violated; FE should be preferred")
print(f"\nFirst 15 rows:")
print(df_firms.head(15))

In [ ]:
# YOUR CODE HERE
# Estimate FE, RE, and run Hausman test
# Then interpret results

# TODO: Your analysis here

In [ ]:
# SOLUTION (hidden)
def analyze_firm_data(df):
    """Complete analysis of firm data."""
    df_panel = df.set_index(['firm_id', 'year'])
    
    # FE
    model_fe = PanelOLS(df_panel['productivity'], df_panel[['rd_spending']], 
                        entity_effects=True)
    results_fe = model_fe.fit(cov_type='clustered', cluster_entity=True)
    
    # RE
    model_re = RandomEffects(df_panel['productivity'], df_panel[['rd_spending']])
    results_re = model_re.fit()
    
    # Hausman test
    beta_fe = results_fe.params['rd_spending']
    beta_re = results_re.params['rd_spending']
    var_fe = results_fe.std_errors['rd_spending']**2
    var_re = results_re.std_errors['rd_spending']**2
    
    hausman = hausman_test_manual(beta_fe, beta_re, var_fe, var_re)
    
    # Print results
    print("Firm R&D and Productivity Analysis")
    print("=" * 70)
    print(f"\n{'Estimator':<20} {'Beta':>12} {'Std Err':>12} {'95% CI':>25}")
    print("-" * 70)
    
    ci_fe = (beta_fe - 1.96*results_fe.std_errors['rd_spending'],
             beta_fe + 1.96*results_fe.std_errors['rd_spending'])
    ci_re = (beta_re - 1.96*results_re.std_errors['rd_spending'],
             beta_re + 1.96*results_re.std_errors['rd_spending'])
    
    print(f"{'True value':<20} {1.2:>12.4f}")
    print(f"{'Fixed Effects':<20} {beta_fe:>12.4f} {results_fe.std_errors['rd_spending']:>12.4f} [{ci_fe[0]:>8.3f}, {ci_fe[1]:>8.3f}]")
    print(f"{'Random Effects':<20} {beta_re:>12.4f} {results_re.std_errors['rd_spending']:>12.4f} [{ci_re[0]:>8.3f}, {ci_re[1]:>8.3f}]")
    
    print("\n" + "=" * 70)
    print("Hausman Test:")
    print("-" * 70)
    print(f"H0: RE assumptions hold")
    print(f"Statistic: {hausman['statistic']:.4f}")
    print(f"P-value:   {hausman['p_value']:.4f}")
    
    if hausman['p_value'] < 0.05:
        print(f"\n⚠️  Reject H0 (p = {hausman['p_value']:.3f} < 0.05)")
        print("\nRECOMMENDATION: Use FIXED EFFECTS")
        print("  - RE is biased (overestimates effect)")
        print("  - Management quality correlated with R&D")
        print(f"  - FE estimate: {beta_fe:.3f} (close to true: 1.2)")
    else:
        print(f"\n✅ Cannot reject H0 (p = {hausman['p_value']:.3f} > 0.05)")
        print("\nRECOMMENDATION: Can use RANDOM EFFECTS")

# Run analysis
analyze_firm_data(df_firms)

---

## Summary

### Key Takeaways

1. **Bias-Efficiency Trade-off:**
   - FE: Always consistent, less efficient
   - RE: More efficient when valid, biased when invalid

2. **Hausman Test:** Tests whether RE assumptions hold
   - Reject → Use FE
   - Fail to reject → Can use RE

3. **Practical Recommendation:** When in doubt, use FE
   - Safer choice (always consistent)
   - Efficiency loss often small with large T

4. **Use RE When:**
   - Hausman test fails to reject
   - Strong theoretical reasons for exogeneity
   - Need time-invariant variable estimates

5. **Use FE When:**
   - Hausman test rejects
   - Endogeneity suspected
   - Within-entity effects are the goal

### Decision Framework Summary

```
Start with theory: Is Cov(mu_i, X) = 0 plausible?
    |
    |-- No/Uncertain --> Use FE
    |
    |-- Yes --> Estimate both, run Hausman test
                    |
                    |-- Reject --> Use FE
                    |-- Fail to reject --> Use RE
```

### What's Next

In Module 4, we'll study additional specification tests and practical model selection issues beyond the Hausman test.

---

**Next:** Module 4 - Model Selection and Specification Tests